In [1]:
import numpy as np
import pandas as pd
from SWMM import SWMM_ENV as SWMM_ENV
from SWMM_GR import SWMM_ENV_GI_utilization as SWMM_ENV_GI
# from PPO import PPO as PPO
from DQN import DQN as DQN
from HC import HC as HC
import datetime
import matplotlib.pyplot as plt
from joblib import Parallel, delayed
from swmm_api import read_out_file
import os
from scipy import stats

C:\Users\tumbl\anaconda3\envs\drainage_new\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Rainfalls

In [2]:
raindata = np.load('rainfall/real_raindata.npy').tolist()

In [3]:
raindata = [
    [
        [time, str(float(rain) * 5)]
        for time, rain in event
    ]
    for event in raindata
]

In [4]:
num_rainfalls = 4

# ENV

In [ ]:
env_params3={
    'orf':'SWMM_GR/chaohu_noHC',
    'orf_save':'chaohu_RTC',
    'parm':'states_yaml/chaohu',
    'advance_seconds':300,
    'kf':1,
    'kc':1,
    'reward_type':'3'
}
env3=SWMM_ENV.SWMM_ENV(env_params3)

env_params4={ 
    'orf':'SWMM/chaohu_noHC', # Without GI
    'orf_save':'chaohu_RTC',
    'parm':'states_yaml/chaohu',
    'advance_seconds':300,
    'kf':1,
    'kc':1,
    'reward_type':'3'
}
env4=SWMM_ENV.SWMM_ENV(env_params4)

In [6]:
env_params_new={
    'orf':'SWMM_GR/chaohu_noHC',
    'orf_save':'chaohu_RTC',
    'parm':'states_yaml/chaohu',
    'advance_seconds':300,
    'kf':1,
    'kc':1,
    'reward_type':'3',
    'run_baseline': True,  
    'run_gi_only': True   
}
env_new=SWMM_ENV_GI.SWMM_ENV(env_params_new)

# No Control

In [7]:
def HC_interact(i,env,rainfall): 
    observation, episode_return, episode_length = env.reset(rainfall[i],i,False,'.'), 0, 0
    storage_states = {'CC-storage':observation[0],'JK-storage':observation[1]}
    done, t= False, 0
    at = [0 for _ in range(7)]
    foreaction=[0 for _ in range(7)]
    while not done:
        observation = np.array(observation).reshape(1, -1)
        at = HC.HC_sample_action(foreaction,storage_states)
        foreaction = at
        observation_new,reward,results,done = env.step(at)
        storage_states = {'CC-storage':observation_new[0],'JK-storage':observation_new[1]}
        t += 1
        observation = observation_new
    return env.results

In [8]:
# No GI
results_HC = {}
for i in range(num_rainfalls):
    # i += 50
    print(f"Processing rainfall {i+1}/{num_rainfalls}")
    results_HC['rainfall'+str(i)] = {}
    results_HC['rainfall'+str(i)]['env4'] = HC_interact(i,env4,raindata)

# GI only
results_HC_GI = {}
for i in range(num_rainfalls):
    # i += 50
    print(f"Processing rainfall {i+1}/{num_rainfalls}")
    results_HC_GI['rainfall'+str(i)] = {}
    results_HC_GI['rainfall'+str(i)]['env3'] = HC_interact(i,env3,raindata)

Processing rainfall 1/4
Processing rainfall 2/4
Processing rainfall 3/4
Processing rainfall 4/4


KeyboardInterrupt: 

In [ ]:
np.save('./results_m_real/HC',results_HC)

np.save('./results_m_real/HC_GI',results_HC_GI)

# DQN

In [7]:
def DQN_interact(model,i,env,raindata):
    observation, episode_return, episode_length = env.reset(raindata[i],i,False,'.'), 0, 0
    done, t= False, 0
    while not done:
        observation = np.array(observation).reshape(1, -1)
        action = DQN.sample_action(observation,model,False)
        at=model.action_table[int(action[0].numpy())].tolist()
        observation_new,reward,results,done = env.step(at)
        observation = observation_new
    return env.results

In [ ]:
agent_params={
    'state_dim':len(env3.config['states']),
    'action_dim':2**len(env3.config['action_assets']),

    'encoding_layer':[50,50,50],
    'value_layer':[50,50,50],
    'advantage_layer':[50,50,50],
    'num_rain':50,

    'train_iterations':20,
    'training_step':1000,
    'gamma':0.3,
    'epsilon':1,
    'ep_min':1e-100,
    'ep_decay':0.999,
    'learning_rate':0.01,

    'action_table':pd.read_csv('SWMM/action_table.csv').values[:,1:],
}

dmodel3 = DQN.DQN(agent_params)   
dmodel3.load_model('./Results_DQN_reward3/model/')

In [9]:
results_DQN_r3 = {}
for i in range(num_rainfalls):
    # i += 50
    print(f"Processing rainfall {i+1}/{num_rainfalls}")
    results_DQN_r3['rainfall'+str(i)] = {}
    results_DQN_r3['rainfall'+str(i)]['env3'] = DQN_interact(dmodel3,i,env3,raindata)

Processing rainfall 1/4
Processing rainfall 2/4
Processing rainfall 3/4
Processing rainfall 4/4


In [ ]:
np.save('./results_m_real/DQN3_GI',results_DQN_r3)

# DQN_new

In [8]:
agent_params={
    'state_dim':len(env_new.config['states']),
    'action_dim':2**len(env_new.config['action_assets']),

    'encoding_layer':[50,50,50],
    'value_layer':[50,50,50],
    'advantage_layer':[50,50,50],
    'num_rain':50,

    'train_iterations':20,
    'training_step':1000,
    'gamma':0.3,
    'epsilon':1,
    'ep_min':1e-100,
    'ep_decay':0.999,
    'learning_rate':0.01,

    'action_table':pd.read_csv('SWMM/action_table.csv').values[:,1:],
}

dmodel3 = DQN.DQN(agent_params)   
dmodel3.load_model('./Results_DQN_reward5/model_82/')

In [9]:
results_DQN_r3_new = {}
for i in range(num_rainfalls):
    # i += 50
    print(f"Processing rainfall {i+1}/{num_rainfalls}")
    results_DQN_r3_new['rainfall'+str(i)] = {}
    results_DQN_r3_new['rainfall'+str(i)]['env_new'] = DQN_interact(dmodel3,i,env_new,raindata)

Processing rainfall 1/4
Processing rainfall 2/4
Processing rainfall 3/4
Processing rainfall 4/4


In [ ]:
np.save('./results_m_real_weights/DQN3_GI_new_82',results_DQN_r3_new)

# HC

In [17]:
env_params5={
    'orf':'SWMM_GR/chaohu',
    'orf_save':'chaohu_RTC',
    'parm':'states_yaml/chaohu',
    'advance_seconds':300,
    'kf':1,
    'kc':1,
    'reward_type':'3'
}
env5=SWMM_ENV.SWMM_ENV(env_params5)

In [18]:
results_HC_GI_true = {}
for i in range(num_rainfalls):
    # i += 50
    print(f"Processing rainfall {i+1}/{num_rainfalls}")
    results_HC_GI_true['rainfall'+str(i)] = {}
    results_HC_GI_true['rainfall'+str(i)]['env5'] = HC_interact(i,env5,raindata)

Processing rainfall 1/4
Processing rainfall 2/4
Processing rainfall 3/4
Processing rainfall 4/4


In [ ]:
np.save('./results_m_real/HC_GI_true',results_HC_GI_true)